# Individual assignment: Using sentiment data to predict future stock market performance
## Oliver Nilsson

In [1]:
import os
import pickle
import pandas as pd
import numpy as np
import spacy
import gc
from tqdm import tqdm
import yfinance as yf
import requests
import time
from urllib3.util.retry import Retry
from requests.adapters import HTTPAdapter
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from langchain.prompts import PromptTemplate
from langchain_ollama import OllamaLLM
import json
import statsmodels.api as sm
from pandas.tseries.offsets import BDay

In [ ]:
# Define the prompt for Ollama
ollama_sentiment_prompt = PromptTemplate(
    template="""
        <|start_header_id|>system<|end_header_id|>

        You should determine if the role is an manager, analyst or other.
        - A manager typically has titles like CEO, CFO, VP, Director, or similar roles that indicate leadership and strategic decision-making.
        - An analyst typically has either a bank name or titles like Analyst, Associate, or similar roles that indicate data analysis and reporting.

        The role is: "{role}"

        Return the role in JSON format. Only "Manager", "Analyst" or "Other" are valid values.

        <|eot_id|>
        <|start_header_id|>assistant<|end_header_id|>
    """,
    input_variables=['role'],
)

# Create the Ollama model
llm = OllamaLLM(model="llama3.1", temperature=0)

# Create the chain
chain = ollama_sentiment_prompt | llm

# Initialize the cache
role_cache = {}
# Load the cache from a file if it exists
cache_file = 'role_cache.json'
if os.path.exists(cache_file):
    with open(cache_file, 'r') as f:
        role_cache = json.load(f)
    print('Loaded role cache from file')

def determine_role(description):
    # Check if the role is already cached
    if description not in role_cache:
        # Run the chain with the role
        sentiment_analysis_result = chain.invoke({'role': description})
        result_json = json.loads(sentiment_analysis_result)
        # Cache the result
        role_cache[description] = result_json['role'].lower()
        # Save the updated cache to a file
        with open('role_cache.json', 'w') as f:
            json.dump(role_cache, f)
    return role_cache[description]

In [ ]:
# Directory where the .p files are stored
directory = 'transcripts/'

# Individual tickers
tickers = ['BX', 'CRM', 'NKE', 'CHTR', 'SE', 'CVS', 'QCOM', 'WFC', 'DIS', 'TSLA', 'GOOG', 'MA', 'PEP', 'VZ', 'TMUS', 'UNP',
            'MS', 'NVS', 'NVO', 'MCO', 'JNJ', 'META', 'MCD', 'GM', 'AMT', 'MRK', 'CVX', 'AMZN', 'FDX', 'AAPL', 'ISRG', 'NVDA',
            'KO', 'LMT', 'SBUX', 'BA', 'HD', 'ADBE', 'MET', 'BMY', 'TD', 'LLY', 'JPM', 'AXP', 'DUK', 'TGT', 'EL', 'TXN', 'F',
            'AMGN', 'INTC', 'ASML', 'ORCL', 'HON', 'SNY', 'C', 'WMT', 'CMCSA', 'EBAY', 'TMO']

# Load the SpaCy small model (less memory usage)
nlp = spacy.load('en_core_web_sm')

# Load and preprocess the Loughran-McDonald sentiment word lists
LM_file_name = 'LoughranMcDonald_SentimentWordLists_2018.xlsx'
LM_negative = pd.read_excel(LM_file_name, sheet_name='Negative', header=None).squeeze().str.lower().tolist()
LM_positive = pd.read_excel(LM_file_name, sheet_name='Positive', header=None).squeeze().str.lower().tolist()
LM_uncertainty = pd.read_excel(LM_file_name, sheet_name='Uncertainty', header=None).squeeze().str.lower().tolist()

# Convert to sets for faster lookups
LM_negative_set = set(LM_negative)
LM_positive_set = set(LM_positive)
LM_uncertainty_set = set(LM_uncertainty)


# Titles to search for in participant descriptions
manager_titles = ['CEO', 'Chief', 'Director', 'Executive', 'President', 'Chairman']
analyst_titles = ['Analyst', 'Research', 'Equity Analyst']

# Function to extract roles from participant descriptions
def count_roles(participants):
    role_counts = {'CEO': 0, 'Analyst': 0, 'Other': 0}

    for participant in participants:
        role = determine_role(participant['description'].lower().strip('.,'))

        # Count the roles
        if role == 'manager':
            role_counts['CEO'] += 1
        elif role == 'analyst':
            role_counts['Analyst'] += 1
        elif role == 'other':
            role_counts['Other'] += 1

    return role_counts

# Define the sentiment scoring function
def score_transcript(text):
    words = text.split()
    number_of_words = len(words)

    if number_of_words == 0:
        return 0.0, 0.0, 0.0

    counts = pd.Series(words).value_counts()
    keys = set(counts.index)

    pos = round((sum([counts[k] for k in (keys & LM_positive_set)]) / number_of_words), 4)
    neg = round((sum([counts[k] for k in (keys & LM_negative_set)]) / number_of_words), 4)
    unc = round((sum([counts[k] for k in (keys & LM_uncertainty_set)]) / number_of_words), 4)

    return pos, neg, unc

# Preprocess text using batch processing in SpaCy to reduce memory load
def preprocess_batch_texts(texts):
    max_length = 10_000  # Limit transcript length to avoid kernel crashes
    processed_texts = []

    docs = nlp.pipe([text[:max_length].lower() for text in texts], batch_size=10)  # Larger batch size for better performance

    for doc in docs:
        words = [token.lemma_ for token in doc if not (token.is_stop or token.is_punct or token.is_space)]
        processed_texts.append(' '.join(words))

    return processed_texts

# Combined function to extract transcript text and roles
def extract_text_and_roles(transcripts):
    texts, times, roles = [], [], []
    for t in transcripts:
        text = ' '.join(speech['speech'][0] for speech in t['transcript'])
        texts.append(text)
        times.append(t['time'])
        roles.append(count_roles(t['participant']))
    return texts, times, roles

# DataFrame to store sentiment scores
sentiment_scores = pd.DataFrame(columns=['date', 'ticker', 'pos', 'neg', 'unc', 'manager_ratio', 'analyst_ratio', 'other_ratio'])

# Initialize progress bar
with tqdm(total=len(tickers), desc="Processing Tickers", unit="ticker") as pbar:

    # Loop through each ticker
    for ticker in tickers:
            file_path = os.path.join(directory, f"{ticker}.p")

            if os.path.exists(file_path):
                with open(file_path, 'rb') as f:
                    transcripts = pickle.load(f)

                if transcripts is not None and len(transcripts) > 0:
                    transcripts_texts, times, roles = extract_text_and_roles(transcripts)

                    if transcripts_texts and times:
                        clean_texts = preprocess_batch_texts(transcripts_texts)

                        for i, clean_text in enumerate(clean_texts):
                            scores = score_transcript(clean_text)
                            role_counts = roles[i]
                            total_roles = sum(role_counts.values())
                            manager_ratio = role_counts['CEO'] / total_roles if total_roles > 0 else 0
                            analyst_ratio = role_counts['Analyst'] / total_roles if total_roles > 0 else 0
                            other_ratio = role_counts['Other'] / total_roles if total_roles > 0 else 0

                            # Append scores and ratios to DataFrame
                            new_row = pd.DataFrame([[pd.to_datetime(times[i]), ticker, *scores, manager_ratio, analyst_ratio, other_ratio]],
                                                    columns=['date', 'ticker', 'pos', 'neg', 'unc', 'manager_ratio', 'analyst_ratio', 'other_ratio'])

                            if sentiment_scores.empty:
                                sentiment_scores = new_row
                            elif (not sentiment_scores.empty) and (not new_row.empty):
                                sentiment_scores = pd.concat([sentiment_scores, new_row]).reset_index(drop=True)

                        gc.collect()  # Memory management after each ticker
                else:
                    print(f"No transcripts found for {ticker}")
            else:
                print(f"File for {ticker} not found!")

            # Update progress bar
            pbar.update(1)

# Check if the DataFrame has any data before writing to the final CSV
if not sentiment_scores.empty:
    sentiment_scores.to_csv('base_sentiment_scores.csv', index=False)   # Save the final sentiment scores
    sentiment_scores.to_csv('sentiment_scores.csv', index=False)        # Save a copy for the next step
    print("Final sentiment scores saved to 'base_sentiment_scores.csv'.")
    print("A copy of the sentiment scores has been saved to 'sentiment_scores.csv'.")
else:
    print("No data found!")

In [2]:
# Load the base data frame from the CSV file
df_base = pd.read_csv('base_sentiment_scores.csv')

def multi_index(data_frame):
    data_frame['date'] = pd.to_datetime(data_frame['date'])
    data_frame.set_index(['ticker', 'date'], inplace=True)
    data_frame.sort_index(inplace=True)

    data_frame.to_csv('sentiment_scores.csv', index=True)   # Save a copy for the next step
    return data_frame

# Usage
sentiment_score = multi_index(df_base)
sentiment_score.head()

pos     neg     unc  manager_ratio  \
ticker date                                                         
AAPL   2006-01-19 13:53:17  0.0206  0.0049  0.0118       0.210526   
       2006-04-20 11:01:29  0.0286  0.0071  0.0051       0.130435   
       2006-07-20 00:49:46  0.0358  0.0082  0.0082       0.130435   
       2006-10-19 04:57:25  0.0362  0.0124  0.0176       0.080000   
       2007-01-18 03:24:19  0.0400  0.0144  0.0044       0.111111   

                            analyst_ratio  other_ratio  
ticker date                                             
AAPL   2006-01-19 13:53:17       0.473684     0.315789  
       2006-04-20 11:01:29       0.608696     0.260870  
       2006-07-20 00:49:46       0.565217     0.304348  
       2006-10-19 04:57:25       0.480000     0.440000  
       2007-01-18 03:24:19       0.481481     0.407407

In [3]:
def add_net_sentiment(data_frame):
    data_frame['net_sentiment'] = data_frame['pos'] - data_frame['neg'] # Calculate net sentiment
    data_frame.to_csv('sentiment_scores.csv')   # Update CSV
    return data_frame

# Usage
sentiment_score = add_net_sentiment(sentiment_score)
sentiment_score

pos     neg     unc  manager_ratio  \
ticker date                                                         
AAPL   2006-01-19 13:53:17  0.0206  0.0049  0.0118       0.210526   
       2006-04-20 11:01:29  0.0286  0.0071  0.0051       0.130435   
       2006-07-20 00:49:46  0.0358  0.0082  0.0082       0.130435   
       2006-10-19 04:57:25  0.0362  0.0124  0.0176       0.080000   
       2007-01-18 03:24:19  0.0400  0.0144  0.0044       0.111111   
...                            ...     ...     ...            ...   
WMT    2020-05-27 22:49:16  0.0251  0.0162  0.0059       0.500000   
       2020-06-04 02:46:45  0.0278  0.0342  0.0063       0.772727   
       2020-08-18 14:54:10  0.0398  0.0205  0.0133       0.142857   
       2020-11-17 19:04:07  0.0513  0.0131  0.0119       0.222222   
       2021-02-20 19:55:06  0.0510  0.0075  0.0050       0.360000   

                            analyst_ratio  other_ratio  net_sentiment  
ticker date                                                            
AAPL   2006-01-19 13:53:17       0.473684     0.315789         0.0157  
       2006-04-20 11:01:29       0.608696     0.260870         0.0215  
       2006-07-20 00:49:46       0.565217     0.304348         0.0276  
       2006-10-19 04:57:25       0.480000     0.440000         0.0238  
       2007-01-18 03:24:19       0.481481     0.407407         0.0256  
...                                   ...          ...            ...  
WMT    2020-05-27 22:49:16       0.000000     0.500000         0.0089  
       2020-06-04 02:46:45       0.000000     0.227273        -0.0064  
       2020-08-18 14:54:10       0.571429     0.285714         0.0193  
       2020-11-17 19:04:07       0.388889     0.388889         0.0382  
       2021-02-20 19:55:06       0.440000     0.200000         0.0435  

[4384 rows x 7 columns]

In [4]:
# Import ticker return and volume data combined with S&P 500 return and close data from WRDS for further analysis
df_ticker_return_volume = pd.read_csv('ticker_return_volume.csv')
df_ticker_return_volume = df_ticker_return_volume.rename(columns={'TICKER': 'ticker', 'VOL': 'volume', 'RET': 'stock_return'})
# Rename ticker "FB" to "META" due to ticker change
df_ticker_return_volume['ticker'] = df_ticker_return_volume['ticker'].replace('FB', 'META')
# Filter df_ticker_return_volume for assigned tickers
tickers = sentiment_score.index.get_level_values('ticker').unique()
df_ticker_return_volume = df_ticker_return_volume[df_ticker_return_volume['ticker'].isin(tickers)]

df_sp500_return_close = pd.read_csv('sp500_return_close.csv')
df_sp500_return_close = df_sp500_return_close.rename(columns={'caldt': 'date', 'sprtrn': 'sp500_return', 'spindx': 'sp500_close'})

df_ticker_sp500 = pd.merge(df_ticker_return_volume, df_sp500_return_close, on='date', how='left')
df_ticker_sp500.drop(columns=['PERMNO'], inplace=True)
df_ticker_sp500 = multi_index(df_ticker_sp500)

# Convert date index to datetime
df_ticker_sp500.index.set_levels(pd.to_datetime(df_ticker_sp500.index.levels[1]), level=1)

# Convert stock_return from object to float64. Some values are strings with "C"
df_ticker_sp500['stock_return'] = pd.to_numeric(df_ticker_sp500['stock_return'], errors='coerce')

df_ticker_sp500.to_csv('ticker_sp500.csv')   # Save a copy for the next step
df_ticker_sp500

volume  stock_return  sp500_return  sp500_close
ticker date                                                           
AAPL   2005-11-16  28207953.0      0.042871      0.001790      1231.21
       2005-11-17  24161976.0     -0.006620      0.009414      1242.80
       2005-11-18  18846012.0      0.000620      0.004401      1248.27
       2005-11-21  18335669.0      0.006196      0.005271      1254.85
       2005-11-22  19365959.0      0.024015      0.005084      1261.23
...                       ...           ...           ...          ...
WMT    2021-03-08  13662797.0     -0.009603     -0.005359      3821.35
       2021-03-09  10662883.0      0.007898      0.014155      3875.44
       2021-03-10  14976448.0      0.025526      0.006030      3898.81
       2021-03-11  11829888.0     -0.000378      0.010395      3939.34
       2021-03-12   9479187.0      0.015061      0.001015      3943.34

[224905 rows x 4 columns]

In [ ]:
def add_market_data(data_frame):
    # Add a temporary column for date only for merging with market data
    data_frame['date_only'] = data_frame.index.get_level_values('date').date

    # Get the start date from the sentiment data
    date_start = data_frame.index.get_level_values('date').min()

    # Get the unique tickers from sentiment_scores
    unique_tickers = data_frame.index.get_level_values('ticker').unique()

    # Initialize empty DataFrame to store market data for all tickers
    market_data_all = pd.DataFrame()

    for ticker in unique_tickers:
        try:
            # Download market data using yfinance
            market_data = yf.download(ticker, start=date_start)
            market_data['returns_market'] = market_data['Adj Close'].pct_change()

            # Convert index to date for matching with sentiment data
            market_data['date'] = market_data.index.date
            market_data['ticker'] = ticker
            market_data.set_index(['ticker', 'date'], inplace=True)
            # Drop subheader level "Ticker"
            market_data.columns = market_data.columns.droplevel(1)

            # Append market data for this ticker to the total market data
            market_data_all = pd.concat([market_data_all, market_data[['Adj Close', 'returns_market']]])
        except Exception as e:
            print(f'Error fetching data for {ticker}: {e}')

    # Merge market data with the sentiment data and fill any missing values
    data_frame = data_frame.merge(market_data_all, left_on=['ticker', 'date_only'], right_index=True, how='left')
    data_frame.drop(columns=['date_only'], inplace=True)  # Drop the temporary date_only column

    # Calculate rolling correlation between net sentiment and returns for each ticker
    window_size = 4
    data_frame['rolling_corr_net_sent'] = np.nan  # Initialize column

    # Group by ticker and calculate rolling correlation within each group
    for ticker, group in data_frame.groupby('ticker'):
        group = group.sort_values(by='date')  # Ensure the data is sorted by date
        rolling_corr = group['returns_market'].rolling(window=window_size).corr(group['net_sentiment'])

        # Update the rolling correlation values in the main DataFrame
        data_frame.loc[(ticker, slice(None)), 'rolling_corr_net_sent'] = rolling_corr

    data_frame.to_csv('sentiment_scores.csv')  # Update CSV
    return data_frame

# Usage
sentiment_score = add_market_data(sentiment_score)
sentiment_score.head()


In [ ]:
def add_sp500_data(data_frame):
    # Add a temporary column for date only for merging with market data
    data_frame['date_only'] = data_frame.index.get_level_values('date').date

    start_date = data_frame.index.get_level_values('date').min()

    # Download S&P 500 data using yfinance
    sp500_data = yf.download('^GSPC', start=start_date)
    sp500_data['returns_sp500'] = sp500_data['Adj Close'].pct_change()

    # Convert index to date for matching with sentiment data
    sp500_data['date'] = sp500_data.index.date
    sp500_data.set_index('date', inplace=True)

    # Drop subheader level "Ticker"
    sp500_data.columns = sp500_data.columns.droplevel(1)

    sp500_data = sp500_data[['Adj Close', 'returns_sp500']]
    sp500_data.rename(columns={'Adj Close': 'Adj Close_SP500'}, inplace=True)

    # Merge market data with the sentiment data and fill any missing values
    data_frame = data_frame.merge(sp500_data, left_on=['date_only'], right_index=True, how='left')
    data_frame.drop(columns=['date_only'], inplace=True)  # Drop the temporary date_only column

    return data_frame


# Usage
sentiment_score = add_sp500_data(sentiment_score)
sentiment_score.head()

In [5]:
def add_lag_returns(data_frame):
    # Calculate lagged returns for the market
    data_frame['lag_stock_return'] = data_frame.groupby('ticker')['stock_return'].shift(1)
    data_frame['lag_sp500_return'] = data_frame.groupby('ticker')['sp500_return'].shift(1)

    return data_frame

# Usage
df_ticker_sp500 = add_lag_returns(df_ticker_sp500)
df_ticker_sp500

volume  stock_return  sp500_return  sp500_close  \
ticker date                                                              
AAPL   2005-11-16  28207953.0      0.042871      0.001790      1231.21   
       2005-11-17  24161976.0     -0.006620      0.009414      1242.80   
       2005-11-18  18846012.0      0.000620      0.004401      1248.27   
       2005-11-21  18335669.0      0.006196      0.005271      1254.85   
       2005-11-22  19365959.0      0.024015      0.005084      1261.23   
...                       ...           ...           ...          ...   
WMT    2021-03-08  13662797.0     -0.009603     -0.005359      3821.35   
       2021-03-09  10662883.0      0.007898      0.014155      3875.44   
       2021-03-10  14976448.0      0.025526      0.006030      3898.81   
       2021-03-11  11829888.0     -0.000378      0.010395      3939.34   
       2021-03-12   9479187.0      0.015061      0.001015      3943.34   

                   lag_stock_return  lag_sp500_return  
ticker date                                            
AAPL   2005-11-16               NaN               NaN  
       2005-11-17          0.042871          0.001790  
       2005-11-18         -0.006620          0.009414  
       2005-11-21          0.000620          0.004401  
       2005-11-22          0.006196          0.005271  
...                             ...               ...  
WMT    2021-03-08          0.012468          0.019496  
       2021-03-09         -0.009603         -0.005359  
       2021-03-10          0.007898          0.014155  
       2021-03-11          0.025526          0.006030  
       2021-03-12         -0.000378          0.010395  

[224905 rows x 6 columns]

In [6]:
# Volatility Column
def add_return_volatility(data_frame):
    # Calculate the rolling standard deviation of returns for each ticker
    window_size = 10
    df_volatility = data_frame.copy()
    #df_volatility.dropna(subset=['returns_market'], inplace=True)  # Drop NaN values in returns_market
    # Calculate the rolling standard deviation of returns for each ticker
    df_volatility['stock_volatility'] = df_volatility.groupby('ticker')['stock_return'].transform(lambda x: x.rolling(window=window_size).std())

    return df_volatility

# Usage
df_ticker_sp500 = add_return_volatility(df_ticker_sp500)
df_ticker_sp500

volume  stock_return  sp500_return  sp500_close  \
ticker date                                                              
AAPL   2005-11-16  28207953.0      0.042871      0.001790      1231.21   
       2005-11-17  24161976.0     -0.006620      0.009414      1242.80   
       2005-11-18  18846012.0      0.000620      0.004401      1248.27   
       2005-11-21  18335669.0      0.006196      0.005271      1254.85   
       2005-11-22  19365959.0      0.024015      0.005084      1261.23   
...                       ...           ...           ...          ...   
WMT    2021-03-08  13662797.0     -0.009603     -0.005359      3821.35   
       2021-03-09  10662883.0      0.007898      0.014155      3875.44   
       2021-03-10  14976448.0      0.025526      0.006030      3898.81   
       2021-03-11  11829888.0     -0.000378      0.010395      3939.34   
       2021-03-12   9479187.0      0.015061      0.001015      3943.34   

                   lag_stock_return  lag_sp500_return  stock_volatility  
ticker date                                                              
AAPL   2005-11-16               NaN               NaN               NaN  
       2005-11-17          0.042871          0.001790               NaN  
       2005-11-18         -0.006620          0.009414               NaN  
       2005-11-21          0.000620          0.004401               NaN  
       2005-11-22          0.006196          0.005271               NaN  
...                             ...               ...               ...  
WMT    2021-03-08          0.012468          0.019496          0.011403  
       2021-03-09         -0.009603         -0.005359          0.011860  
       2021-03-10          0.007898          0.014155          0.014434  
       2021-03-11          0.025526          0.006030          0.014102  
       2021-03-12         -0.000378          0.010395          0.013636  

[224905 rows x 7 columns]

In [7]:
df_ticker_sp500.dtypes

volume              float64
stock_return        float64
sp500_return        float64
sp500_close         float64
lag_stock_return    float64
lag_sp500_return    float64
stock_volatility    float64
dtype: object

In [9]:
# Merge sentiment_score and df_ticker_sp500 dataframes

# Sort both dataframes by index for merge_asof
sentiment_score = sentiment_score.sort_index(level='date')
df_ticker_sp500 = df_ticker_sp500.sort_index(level='date')

# Use merge_asof to match on 'date' with tolerance to ignore time differences
df_main = pd.merge_asof(
    sentiment_score.reset_index(),
    df_ticker_sp500.reset_index(),
    on='date',
    by='ticker',
    direction='backward'
)

# Set the original index if needed
df_main = multi_index(df_main)

df_main

pos     neg     unc  manager_ratio  \
ticker date                                                         
AAPL   2006-01-19 13:53:17  0.0206  0.0049  0.0118       0.210526   
       2006-04-20 11:01:29  0.0286  0.0071  0.0051       0.130435   
       2006-07-20 00:49:46  0.0358  0.0082  0.0082       0.130435   
       2006-10-19 04:57:25  0.0362  0.0124  0.0176       0.080000   
       2007-01-18 03:24:19  0.0400  0.0144  0.0044       0.111111   
...                            ...     ...     ...            ...   
WMT    2020-05-27 22:49:16  0.0251  0.0162  0.0059       0.500000   
       2020-06-04 02:46:45  0.0278  0.0342  0.0063       0.772727   
       2020-08-18 14:54:10  0.0398  0.0205  0.0133       0.142857   
       2020-11-17 19:04:07  0.0513  0.0131  0.0119       0.222222   
       2021-02-20 19:55:06  0.0510  0.0075  0.0050       0.360000   

                            analyst_ratio  other_ratio  net_sentiment  \
ticker date                                                             
AAPL   2006-01-19 13:53:17       0.473684     0.315789         0.0157   
       2006-04-20 11:01:29       0.608696     0.260870         0.0215   
       2006-07-20 00:49:46       0.565217     0.304348         0.0276   
       2006-10-19 04:57:25       0.480000     0.440000         0.0238   
       2007-01-18 03:24:19       0.481481     0.407407         0.0256   
...                                   ...          ...            ...   
WMT    2020-05-27 22:49:16       0.000000     0.500000         0.0089   
       2020-06-04 02:46:45       0.000000     0.227273        -0.0064   
       2020-08-18 14:54:10       0.571429     0.285714         0.0193   
       2020-11-17 19:04:07       0.388889     0.388889         0.0382   
       2021-02-20 19:55:06       0.440000     0.200000         0.0435   

                                volume  stock_return  sp500_return  \
ticker date                                                          
AAPL   2006-01-19 13:53:17  60724482.0     -0.041885      0.005564   
       2006-04-20 11:01:29  59592071.0      0.030160      0.001168   
       2006-07-20 00:49:46  70582838.0      0.118299     -0.008477   
       2006-10-19 04:57:25  54188273.0      0.059842      0.000732   
       2007-01-18 03:24:19  84566630.0     -0.061927     -0.002971   
...                                ...           ...           ...   
WMT    2020-05-27 22:49:16  10357895.0     -0.011142      0.014827   
       2020-06-04 02:46:45   8004963.0     -0.011015     -0.003369   
       2020-08-18 14:54:10  26744177.0     -0.006563      0.002303   
       2020-11-17 19:04:07  14237213.0     -0.020139     -0.004792   
       2021-02-20 19:55:06  12201992.0      0.004940     -0.001855   

                            sp500_close  lag_stock_return  lag_sp500_return  \
ticker date                                                                   
AAPL   2006-01-19 13:53:17      1285.04         -0.026207         -0.003897   
       2006-04-20 11:01:29      1311.46         -0.008608          0.001744   
       2006-07-20 00:49:46      1249.13          0.022684          0.018555   
       2006-10-19 04:57:25      1366.96          0.003231          0.001400   
       2007-01-18 03:24:19      1426.37         -0.022142         -0.000894   
...                                 ...               ...               ...   
WMT    2020-05-27 22:49:16      3036.13         -0.003780          0.012289   
       2020-06-04 02:46:45      3112.35         -0.003792          0.013649   
       2020-08-18 14:54:10      3389.78          0.022624          0.002710   
       2020-11-17 19:04:07      3609.53          0.012621          0.011648   
       2021-02-20 19:55:06      3906.71         -0.064810         -0.004416   

                            stock_volatility  
ticker date                                   
AAPL   2006-01-19 13:53:17          0.030969  
       2006-04-20 11:01:29          0.027492  
       2006-07-20 00:49:46          0.045439  
       20

In [10]:
# Function to calculate cumulative abnormal returns (CAR) for a specific ticker and date
def calc_abnormal_returns(row):
    # Extract ticker and date from the MultiIndex of the current row
    ticker, date = row.name

    # Normalize the date to avoid time differences
    date = pd.to_datetime(date).normalize()
    
    # Filter and extract the calculation data for the specific ticker
    df_ticker_sp500_filter = df_ticker_sp500.loc[ticker].copy()

    # Step 1: Define estimation window (30 days before the event)
    start_date = date - BDay(30)    # Start 30 business days before event
    end_date = date - BDay(1)       # End 1 business day before event
    estimation_window_data = df_ticker_sp500_filter.loc[start_date:end_date]

    # Step 2: Define event window (e.g., [-1, +1] days around the earnings call date)
    event_window_data = df_ticker_sp500_filter.loc[date - BDay(1):date + BDay(1)]

    # Ensure we have enough data in the estimation window
    if len(estimation_window_data) < 15:  # Minimum threshold for estimation window
        print(f"Not enough data for estimation window for {ticker} on {date}")
        return np.nan  # Return NaN for insufficient data

    # Step 3: Estimate alpha and beta using OLS regression in the estimation window
    X = sm.add_constant(estimation_window_data['sp500_return'])  # Market returns with intercept
    y = estimation_window_data['stock_return']
    market_model = sm.OLS(y, X).fit()
    alpha, beta = market_model.params

    # Step 4: Calculate abnormal returns in the event window
    event_window_data = event_window_data.copy()  # Avoid SettingWithCopyWarning
    event_window_data['expected_return'] = alpha + beta * event_window_data['sp500_return']
    event_window_data['abnormal_return'] = event_window_data['stock_return'] - event_window_data['expected_return']

    # Calculate CAR by summing abnormal returns over the event window
    CAR = event_window_data['abnormal_return'].sum()
    return CAR

# Apply function for each row in the DataFrame
df_main['CAR'] = df_main.apply(calc_abnormal_returns, axis=1)
df_main


Not enough data for estimation window for CHTR on 2010-01-28 00:00:00
Not enough data for estimation window for CMCSA on 2005-11-16 00:00:00
Not enough data for estimation window for CRM on 2005-11-24 00:00:00
Not enough data for estimation window for EL on 2005-11-01 00:00:00
Not enough data for estimation window for QCOM on 2005-11-16 00:00:00
Not enough data for estimation window for TMUS on 2007-11-14 00:00:00
Not enough data for estimation window for TMUS on 2008-03-04 00:00:00
Not enough data for estimation window for TMUS on 2008-05-06 00:00:00
Not enough data for estimation window for TMUS on 2008-08-07 00:00:00
Not enough data for estimation window for TMUS on 2008-11-05 00:00:00
Not enough data for estimation window for TMUS on 2009-02-26 00:00:00
Not enough data for estimation window for TMUS on 2009-05-07 00:00:00
Not enough data for estimation window for TMUS on 2009-11-05 00:00:00
Not enough data for estimation window for TMUS on 2010-11-04 00:00:00
Not enough data for es

pos     neg     unc  manager_ratio  \
ticker date                                                         
AAPL   2006-01-19 13:53:17  0.0206  0.0049  0.0118       0.210526   
       2006-04-20 11:01:29  0.0286  0.0071  0.0051       0.130435   
       2006-07-20 00:49:46  0.0358  0.0082  0.0082       0.130435   
       2006-10-19 04:57:25  0.0362  0.0124  0.0176       0.080000   
       2007-01-18 03:24:19  0.0400  0.0144  0.0044       0.111111   
...                            ...     ...     ...            ...   
WMT    2020-05-27 22:49:16  0.0251  0.0162  0.0059       0.500000   
       2020-06-04 02:46:45  0.0278  0.0342  0.0063       0.772727   
       2020-08-18 14:54:10  0.0398  0.0205  0.0133       0.142857   
       2020-11-17 19:04:07  0.0513  0.0131  0.0119       0.222222   
       2021-02-20 19:55:06  0.0510  0.0075  0.0050       0.360000   

                            analyst_ratio  other_ratio  net_sentiment  \
ticker date                                                             
AAPL   2006-01-19 13:53:17       0.473684     0.315789         0.0157   
       2006-04-20 11:01:29       0.608696     0.260870         0.0215   
       2006-07-20 00:49:46       0.565217     0.304348         0.0276   
       2006-10-19 04:57:25       0.480000     0.440000         0.0238   
       2007-01-18 03:24:19       0.481481     0.407407         0.0256   
...                                   ...          ...            ...   
WMT    2020-05-27 22:49:16       0.000000     0.500000         0.0089   
       2020-06-04 02:46:45       0.000000     0.227273        -0.0064   
       2020-08-18 14:54:10       0.571429     0.285714         0.0193   
       2020-11-17 19:04:07       0.388889     0.388889         0.0382   
       2021-02-20 19:55:06       0.440000     0.200000         0.0435   

                                volume  stock_return  sp500_return  \
ticker date                                                          
AAPL   2006-01-19 13:53:17  60724482.0     -0.041885      0.005564   
       2006-04-20 11:01:29  59592071.0      0.030160      0.001168   
       2006-07-20 00:49:46  70582838.0      0.118299     -0.008477   
       2006-10-19 04:57:25  54188273.0      0.059842      0.000732   
       2007-01-18 03:24:19  84566630.0     -0.061927     -0.002971   
...                                ...           ...           ...   
WMT    2020-05-27 22:49:16  10357895.0     -0.011142      0.014827   
       2020-06-04 02:46:45   8004963.0     -0.011015     -0.003369   
       2020-08-18 14:54:10  26744177.0     -0.006563      0.002303   
       2020-11-17 19:04:07  14237213.0     -0.020139     -0.004792   
       2021-02-20 19:55:06  12201992.0      0.004940     -0.001855   

                            sp500_close  lag_stock_return  lag_sp500_return  \
ticker date                                                                   
AAPL   2006-01-19 13:53:17      1285.04         -0.026207         -0.003897   
       2006-04-20 11:01:29      1311.46         -0.008608          0.001744   
       2006-07-20 00:49:46      1249.13          0.022684          0.018555   
       2006-10-19 04:57:25      1366.96          0.003231          0.001400   
       2007-01-18 03:24:19      1426.37         -0.022142         -0.000894   
...                                 ...               ...               ...   
WMT    2020-05-27 22:49:16      3036.13         -0.003780          0.012289   
       2020-06-04 02:46:45      3112.35         -0.003792          0.013649   
       2020-08-18 14:54:10      3389.78          0.022624          0.002710   
       2020-11-17 19:04:07      3609.53          0.012621          0.011648   
       2021-02-20 19:55:06      3906.71         -0.064810         -0.004416   

                            stock_volatility       CAR  
ticker date                                             
AAPL   2006-01-19 13:53:17          0.030969 -0.093853  
       2006-04-20 11:01:29          0.027492  0.011100  
       2006-07-2

In [11]:
# Pint sum of rows of the DataFrame df_main
print(df_main.shape)

(4384, 15)


In [12]:
def add_sector(data_frame):
    # Get the industry sector for each ticker
    unique_tickers = data_frame.index.get_level_values('ticker').unique()
    ticker_sector = {}

    # Fetch industry sector for each ticker
    for ticker in unique_tickers:
        try:
            stock = yf.Ticker(ticker)
            info = stock.info
            sector = info.get('sector', np.nan)
            ticker_sector[ticker] = sector
        except Exception as e:
            ticker_sector[ticker] = np.nan
            print(f'Error getting info for {ticker}: {e}')

    # Map the industry sector to each ticker in the sentiment_scores DataFrame
    data_frame['sector'] = data_frame.index.get_level_values('ticker').map(ticker_sector)

    # Add sector as multi index level: sector, ticker, date
    data_frame.set_index('sector', append=True, inplace=True)
    data_frame = data_frame.reorder_levels(['sector', 'ticker', 'date'])

    return data_frame

# Usage
df_main = add_sector(df_main)
df_main

pos     neg     unc  \
sector             ticker date                                          
Technology         AAPL   2006-01-19 13:53:17  0.0206  0.0049  0.0118   
                          2006-04-20 11:01:29  0.0286  0.0071  0.0051   
                          2006-07-20 00:49:46  0.0358  0.0082  0.0082   
                          2006-10-19 04:57:25  0.0362  0.0124  0.0176   
                          2007-01-18 03:24:19  0.0400  0.0144  0.0044   
...                                               ...     ...     ...   
Consumer Defensive WMT    2020-05-27 22:49:16  0.0251  0.0162  0.0059   
                          2020-06-04 02:46:45  0.0278  0.0342  0.0063   
                          2020-08-18 14:54:10  0.0398  0.0205  0.0133   
                          2020-11-17 19:04:07  0.0513  0.0131  0.0119   
                          2021-02-20 19:55:06  0.0510  0.0075  0.0050   

                                               manager_ratio  analyst_ratio  \
sector             ticker date                                                
Technology         AAPL   2006-01-19 13:53:17       0.210526       0.473684   
                          2006-04-20 11:01:29       0.130435       0.608696   
                          2006-07-20 00:49:46       0.130435       0.565217   
                          2006-10-19 04:57:25       0.080000       0.480000   
                          2007-01-18 03:24:19       0.111111       0.481481   
...                                                      ...            ...   
Consumer Defensive WMT    2020-05-27 22:49:16       0.500000       0.000000   
                          2020-06-04 02:46:45       0.772727       0.000000   
                          2020-08-18 14:54:10       0.142857       0.571429   
                          2020-11-17 19:04:07       0.222222       0.388889   
                          2021-02-20 19:55:06       0.360000       0.440000   

                                               other_ratio  net_sentiment  \
sector             ticker date                                              
Technology         AAPL   2006-01-19 13:53:17     0.315789         0.0157   
                          2006-04-20 11:01:29     0.260870         0.0215   
                          2006-07-20 00:49:46     0.304348         0.0276   
                          2006-10-19 04:57:25     0.440000         0.0238   
                          2007-01-18 03:24:19     0.407407         0.0256   
...                                                    ...            ...   
Consumer Defensive WMT    2020-05-27 22:49:16     0.500000         0.0089   
                          2020-06-04 02:46:45     0.227273        -0.0064   
                          2020-08-18 14:54:10     0.285714         0.0193   
                          2020-11-17 19:04:07     0.388889         0.0382   
                          2021-02-20 19:55:06     0.200000         0.0435   

                                                   volume  stock_return  \
sector             ticker date                                            
Technology         AAPL   2006-01-19 13:53:17  60724482.0     -0.041885   
                          2006-04-20 11:01:29  59592071.0      0.030160   
                          2006-07-20 00:49:46  70582838.0      0.118299   
                          2006-10-19 04:57:25  54188273.0      0.059842   
                          2007-01-18 03:24:19  84566630.0     -0.061927   
...                                                   ...           ...   
Consumer Defensive WMT    2020-05-27 22:49:16  10357895.0     -0.011142   
                          2020-06-04 02:46:45   8004963.0     -0.011015   
                          2020-08-18 14:54:10  26744177.0     -0.006563   
                          2020-11-17 19:04:07  14237213.0     -0.020139   
                          2021-02-20 19:55:06  12201992.0      0.004940   

                                               sp500_return  sp500_close  \
sector            

In [13]:
def add_sector_ticker_avg_sentiment(data_frame):
    # Calculate average net sentiment for each ticker and sector
    data_frame['ticker_avg_net_sentiment'] = data_frame.groupby('ticker')['net_sentiment'].transform('mean')
    data_frame['sector_avg_net_sentiment'] = data_frame.groupby('sector')['net_sentiment'].transform('mean')
    return data_frame

# Usage
df_main = add_sector_ticker_avg_sentiment(df_main)

# Print only the sector "Technology"
df_tech = df_main.loc['Technology'].reset_index()[['date', 'ticker', 'net_sentiment', 'ticker_avg_net_sentiment', 'sector_avg_net_sentiment']]

In [14]:
df_main.dtypes

pos                         float64
neg                         float64
unc                         float64
manager_ratio               float64
analyst_ratio               float64
other_ratio                 float64
net_sentiment               float64
volume                      float64
stock_return                float64
sp500_return                float64
sp500_close                 float64
lag_stock_return            float64
lag_sp500_return            float64
stock_volatility            float64
CAR                         float64
ticker_avg_net_sentiment    float64
sector_avg_net_sentiment    float64
dtype: object

In [ ]:
tickers = ['BX', 'CRM', 'NKE', 'CHTR', 'SE', 'CVS', 'QCOM', 'WFC', 'DIS', 'TSLA', 'GOOG', 'MA', 'PEP', 'VZ', 'TMUS', 'UNP',
            'MS', 'NVS', 'NVO', 'MCO', 'JNJ', 'META', 'MCD', 'GM', 'AMT', 'MRK', 'CVX', 'AMZN', 'FDX', 'AAPL', 'ISRG', 'NVDA',
            'KO', 'LMT', 'SBUX', 'BA', 'HD', 'ADBE', 'MET', 'BMY', 'TD', 'LLY', 'JPM', 'AXP', 'DUK', 'TGT', 'EL', 'TXN', 'F',
            'AMGN', 'INTC', 'ASML', 'ORCL', 'HON', 'SNY', 'C', 'WMT', 'CMCSA', 'EBAY', 'TMO']

# Print all tickers with space between on one row
print(' '.join(tickers))

In [ ]:
print(sentiment_score.index.get_level_values('date').min())
print(sentiment_score.index.get_level_values('date').max())